# 🎯 [OPS02_M03] 최단거리 매칭 기반 단일 항적-센서 경보 1차 융합(Fusion) 알고리즘 구현
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M03 |
| **미션 주제** | 최단거리 매칭 기반 단일 항적-센서 경보 1차 융합(Fusion) 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 최단거리 매칭 기반 단일 항적-센서 경보 1차 융합(Fusion) 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

최단 거리(Euclidean/Mahalanobis) 매칭 기반으로 동일 비행체에서 발생한 다중 센서 경보를 1차 클러스터링하는 항적 융합(Fusion) Baseline 로직을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **매 스캔마다 여러 센서 경보를 최단거리(유클리드·마할라노비스) 매칭으로 묶어 비행체 1개당 1개의 융합 경보를 만드는 Baseline을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **정밀도가 높은 센서 경보를 기준으로 다른 센서 경보를 거리 게이트 안에서 묶는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/ops02_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건

- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성


- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성


- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""매 스캔(1초)마다 여러 센서 경보를 최단거리(유클리드/마할라노비스) 매칭으로 묶어 비행체 1개당 1개의 융합 경보로 만드는 1차 항적 융합(Fusion) Baseline을 구현하고, 통합 정확도와 한계를 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 다중 센서(레이더·EO/IR·음향) 경보 생성기 (실제 장비 제원·배치와 무관)
# 가상 방어 지역(원점 기준 동/북 km)에 비행체 궤적을 만들고, 센서마다 '자기 위치 기준
# 거리·방위'와 '자기 시계 기준 시각'으로 경보를 낸다. 같은 비행체가 여러 센서에 동시에
# 잡히므로 경보가 중복되고, 센서마다 시계 오차·측정 오차·오경보가 다르다.
# ---------------------------------------------------------------------------
SENSORS = {
    "RDR-A": {"type": "radar", "pos": (0.0, 0.0), "sr": 0.08, "saz": 0.4, "pd": 0.95, "fa": 1.0, "max_r": 40.0, "clock": 0.0},
    "RDR-B": {"type": "radar", "pos": (16.0, -6.0), "sr": 0.10, "saz": 0.5, "pd": 0.93, "fa": 1.0, "max_r": 40.0, "clock": 0.42},
    "EOIR-C": {"type": "eoir", "pos": (5.0, 9.0), "sr": 0.40, "saz": 0.15, "pd": 0.88, "fa": 0.5, "max_r": 18.0, "clock": -0.65},
    "ACU-D": {"type": "acoustic", "pos": (10.0, 2.0), "sr": 0.60, "saz": 2.0, "pd": 0.80, "fa": 0.6, "max_r": 9.0, "clock": 0.93},
}
CENTER = np.array([8.0, 2.0])                         # 가상 방어 지역 중심
KINDS = {"uav": (0.03, 0.06), "heli": (0.04, 0.07), "fast": (0.15, 0.20)}   # 속도 범위(km/s, 추상값)
WEATHER = {"clear": {"eoir": 1.0, "acoustic": 1.0, "radar": 1.0, "fa": 1.0},
           "night": {"eoir": 0.9, "acoustic": 1.0, "radar": 1.0, "fa": 1.1},
           "fog": {"eoir": 0.55, "acoustic": 1.0, "radar": 1.0, "fa": 1.2},
           "rain": {"eoir": 0.7, "acoustic": 0.6, "radar": 0.97, "fa": 1.6}}


def _targets(rng, n, T, special):
    out, kinds = {}, {}
    for i in range(n):
        kind = "fast" if special == "fast" and i < 2 else str(rng.choice(["uav", "uav", "heli", "fast"]))
        ang = rng.uniform(0, 2 * np.pi)
        p = CENTER + rng.uniform(6, 12) * np.array([np.sin(ang), np.cos(ang)])
        hd = np.arctan2(*(CENTER - p)) + rng.normal(0, 0.35)          # 방어 지역 쪽으로(북 기준 방위)
        sp = rng.uniform(*KINDS[kind])
        omega = rng.uniform(0.04, 0.08) * rng.choice([-1, 1]) if (special == "turn" and i < 3) or rng.random() < 0.25 else 0.0
        if special == "formation" and i == 0:
            omega = 0.0
        if special == "cross" and i < 2:                              # 두 표적이 중심에서 교차
            p = CENTER + np.array([-6.0, 0.0]) if i == 0 else CENTER + np.array([0.0, -6.0])
            hd, sp, omega, kind = (np.pi / 2 if i == 0 else 0.0), 0.1, 0.0, "heli"
        if special == "formation" and i == 1:                         # 0.4 km 간격 편대
            p0 = out["TGT00"][0]
            p, hd, sp, omega, kind = p0[:2] + np.array([0.4, 0.0]), np.arctan2(p0[2], p0[3]), np.hypot(p0[2], p0[3]), 0.0, kinds["TGT00"]
        traj = np.zeros((T, 4))
        for t in range(T):
            if t > 0:
                if 20 <= t < 40:
                    hd += omega                                       # 20~40초 선회 기동
                p = p + sp * np.array([np.sin(hd), np.cos(hd)])
            traj[t] = [p[0], p[1], sp * np.sin(hd), sp * np.cos(hd)]
        out[f"TGT{i:02d}"], kinds[f"TGT{i:02d}"] = traj, kind
    return out, kinds


def make_scenario(seed, n=6, T=60, weather="clear", fa_mult=1.0, dropout=None, ghost=False, special=None,
                  noise_mult=None, drift=0.0):
    """시나리오 1개: 정답 궤적, 센서 경보(보고) 목록, 시계 동기화용 기준 신호(heartbeat)를 만든다.
    dropout=(센서, 시작, 끝): 통신 단절, ghost: 한 센서에만 보이는 거짓 항적(기만·스푸핑),
    noise_mult={센서: 배수}: 재밍으로 측정 잡음 증가, drift: 음향 센서 시계가 초당 drift초씩 흐트러짐."""
    rng = np.random.default_rng(seed)
    targets, kinds = _targets(rng, n, T, special)
    w = WEATHER[weather]
    reports, hb, mid = [], [], 0
    gpos = CENTER + np.array([-9.0, 7.0])
    for t in range(T):
        for sid, s in SENSORS.items():
            clock = s["clock"] + (drift * t if sid == "ACU-D" else 0.0)
            if t % 10 == 0:
                hb.append({"sensor": sid, "t_ref": float(t), "t_sensor": t + clock + rng.normal(0, 0.02)})
            if dropout and sid == dropout[0] and dropout[1] <= t < dropout[2]:
                continue
            nm = (noise_mult or {}).get(sid, 1.0)
            objs = [(tid, tr[t, :2]) for tid, tr in targets.items()]
            if ghost and sid == "EOIR-C":
                objs.append(("GHOST", gpos + 0.12 * t * np.array([0.6, -0.8])))
            for tid, xy in objs:
                d = xy - np.array(s["pos"])
                r = float(np.hypot(*d))
                if r > s["max_r"] or rng.random() > s["pd"] * w[s["type"]]:
                    continue
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(r + rng.normal(0, s["sr"] * nm), 3),
                                "az_deg": round(float((np.degrees(np.arctan2(d[0], d[1])) + rng.normal(0, s["saz"] * nm)) % 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": tid})
                mid += 1
            for _ in range(rng.poisson(s["fa"] * fa_mult * w["fa"] * (3 if nm > 1 else 1))):   # 오경보(클러터·반사)
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(rng.uniform(0.5, s["max_r"]), 3), "az_deg": round(rng.uniform(0, 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": "FA"})
                mid += 1
    return {"targets": targets, "kinds": kinds, "reports": reports, "heartbeats": hb, "T": T, "weather": weather}
# ---------------------------------------------------------------------------

# ---- 시공간 전처리 공통 함수: 정제 -> 시각 동기화 -> 공통 좌표(ENU, km) 변환 ----
def estimate_offsets(heartbeats):
    """센서별 시계 오차 = 기준 신호(heartbeat) 대비 센서 시각 차이의 중앙값(잡음에 강함)."""
    df = pd.DataFrame(heartbeats)
    return (df["t_sensor"] - df["t_ref"]).groupby(df["sensor"]).median().to_dict()


def clean(reports):
    """중복 메시지·결측·범위 밖(반사파 등 최대 탐지거리 초과) 경보를 걸러낸다. (정제 목록, 사유별 건수)"""
    seen, out, why = set(), [], {"중복": 0, "결측": 0, "범위 밖": 0}
    for r in reports:
        if r["msg_id"] in seen:
            why["중복"] += 1; continue
        seen.add(r["msg_id"])
        if any(r.get(k) is None or (isinstance(r.get(k), float) and np.isnan(r[k])) for k in ("t_sensor", "range_km", "az_deg")):
            why["결측"] += 1; continue
        if not 0 <= r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1:
            why["범위 밖"] += 1; continue
        out.append(r)
    return out, why


def to_enu(r, offsets):
    """센서 국소 극좌표(거리·방위) -> 공통 ENU(동·북 km), 센서 시각 -> 기준 시각."""
    s, a = SENSORS[r["sensor"]], np.radians(r["az_deg"])
    return {**r, "t": r["t_sensor"] - offsets.get(r["sensor"], 0.0),
            "x": s["pos"][0] + r["range_km"] * np.sin(a), "y": s["pos"][1] + r["range_km"] * np.cos(a)}


def meas_cov(r):
    """거리·방위 오차를 ENU 공분산(2x2)으로 변환한다(1차 근사)."""
    a, rr = np.radians(r["az_deg"]), r["range_km"]
    J = np.array([[np.sin(a), rr * np.cos(a)], [np.cos(a), -rr * np.sin(a)]])
    return J @ np.diag([r["sr"] ** 2, np.radians(r["saz"]) ** 2]) @ J.T


def prepare(scn, sync=True):
    reps, _ = clean(scn["reports"])
    off = estimate_offsets(scn["heartbeats"]) if sync else {}
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"])


def fuse_scan(reps, gate, metric):
    """정밀한 센서 경보부터 차례로, 같은 센서가 아직 없는 가장 가까운 묶음에 넣는다(게이트 밖이면 새 묶음).
    묶음 위치는 측정 공분산의 역수(정보량) 가중 평균으로 갱신한다."""
    clusters = []
    for r in sorted(reps, key=lambda r: SENSORS[r["sensor"]]["sr"]):
        p, C = np.array([r["x"], r["y"]]), meas_cov(r)
        best, bd = None, gate
        for c in clusters:
            if r["sensor"] in c["sensors"]:
                continue
            d = p - c["pos"]
            dist = np.hypot(*d) if metric == "euclid" else np.sqrt(d @ np.linalg.solve(C + c["cov"], d))
            if dist < bd:
                best, bd = c, dist
        if best is None:
            clusters.append({"pos": p, "cov": C, "sensors": {r["sensor"]}, "members": [r], "dist": []})
        else:
            info = np.linalg.inv(best["cov"]) + np.linalg.inv(C)
            best["pos"] = np.linalg.solve(info, np.linalg.solve(best["cov"], best["pos"]) + np.linalg.solve(C, p))
            best["cov"] = np.linalg.inv(info)
            best["sensors"].add(r["sensor"]); best["members"].append(r); best["dist"].append(round(float(bd), 2))
    return clusters


def evaluate_scans(reps, gate, metric):
    """통합 정확도 = (스캔, 실제 비행체) 중 정확히 1개의 순수한 묶음으로 통합된 비율."""
    by = {}
    for r in reps:
        by.setdefault(int(round(r["t"])), []).append(r)
    ok = tot = n_cl = n_multi_fa = 0
    t0 = time.perf_counter()
    for k, rs in by.items():
        cl = fuse_scan(rs, gate, metric)
        n_cl += len(cl)
        n_multi_fa += sum(1 for c in cl if len(c["sensors"]) >= 2 and all(m["truth"] == "FA" for m in c["members"]))
        for g in {r["truth"] for r in rs if r["truth"] != "FA"}:
            hold = [c for c in cl if any(m["truth"] == g for m in c["members"])]
            tot += 1
            ok += len(hold) == 1 and all(m["truth"] == g for m in hold[0]["members"])
    ms = (time.perf_counter() - t0) * 1000 / len(by)
    return {"경보": len(reps), "융합 묶음": n_cl, "통합 정확도": round(ok / tot, 3), "다중센서 거짓묶음": n_multi_fa, "스캔당 ms": round(ms, 2)}


# [교육용가상자료] 기상·기동이 다른 가상 시나리오 4개
scns = [make_scenario(30 + i, n=6, weather=w, special=sp) for i, (w, sp) in enumerate([("clear", None), ("rain", None), ("clear", "turn"), ("fog", "formation")])]
reps = [prepare(s) for s in scns]

rows = []
for name, gate, metric in [("유클리드 1.0km", 1.0, "euclid"), ("마할라노비스 3.0", 3.0, "mahal")]:
    for s, r in zip(scns, reps):
        rows.append({"방식": name, "기상": s["weather"], **evaluate_scans(r, gate, metric)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
summ = df.groupby("방식")[["통합 정확도", "스캔당 ms"]].mean().round(3)
print("\n" + summ.to_string())
best = summ["통합 정확도"].max()
print(f"\n목표(통합 정확도 75% 이상) 달성 여부: {'달성' if best >= 0.75 else '미달'} (최고 {best:.1%})")

# 게이트 크기 민감도(맑음 시나리오)
print("\n[게이트 민감도] 작으면 같은 비행체가 쪼개지고, 크면 다른 비행체·오경보가 섞인다")
for metric, gates in [("euclid", [0.3, 0.6, 1.0, 1.5, 2.5]), ("mahal", [1.5, 2.0, 3.0, 4.0, 6.0])]:
    print(metric, " ".join(f"{g}:{evaluate_scans(reps[0], g, metric)['통합 정확도']:.3f}" for g in gates))

# 시각 동기화 효과
e0 = evaluate_scans(prepare(scns[2], sync=False), 3.0, "mahal")["통합 정확도"]
print(f"\n시각 동기화 생략 시(기동 시나리오) 통합 정확도 {e0:.3f} -> 동기화 적용 {df.iloc[6]['통합 정확도']:.3f}")

# 판단 근거(투명성): 한 스캔의 융합 결과
k = 25
cl = fuse_scan([r for r in reps[0] if int(round(r["t"])) == k], 3.0, "mahal")
print(f"\n[스캔 {k}초 융합 결과 예시: 묶음 {len(cl)}개]")
for c in sorted(cl, key=lambda c: -len(c["sensors"]))[:5]:
    print(f"위치({c['pos'][0]:6.2f},{c['pos'][1]:6.2f}) 센서 {sorted(c['sensors'])} 결합 거리 {c['dist']} 정답 {sorted({m['truth'] for m in c['members']})}")
print("한계: 스캔마다 독립적으로 묶어 속도 정보를 쓰지 않으므로, 가까운 편대·교차 표적과 단일 센서 오경보를 구분하기 어렵다(미션 4에서 칼만 필터로 보완).")

# 그림: 원시 경보 vs 융합 묶음
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
colors = dict(zip(SENSORS, ["tab:blue", "tab:green", "tab:orange", "tab:red"]))
for r in [r for r in reps[0] if int(round(r["t"])) == k]:
    ax[0].scatter(r["x"], r["y"], color=colors[r["sensor"]], s=18)
for sid, c in colors.items():
    ax[0].scatter([], [], color=c, label=sid)
for c in cl:
    ax[1].scatter(*c["pos"], s=40 + 40 * len(c["sensors"]), color="tab:purple" if len(c["sensors"]) > 1 else "lightgray", edgecolor="k")
for g, tr in scns[0]["targets"].items():
    ax[1].plot(tr[k, 0], tr[k, 1], "k+", ms=10)
ax[0].set_title(f"Raw alerts at t={k}s ({sum(1 for r in reps[0] if int(round(r['t'])) == k)})"); ax[0].legend(fontsize=8)
ax[1].set_title(f"Fused clusters ({len(cl)}), + = truth, size = #sensors")
for a in ax:
    a.set_xlabel("East (km)"); a.set_ylabel("North (km)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
        방식    기상   경보  융합 묶음  통합 정확도  다중센서 거짓묶음  스캔당 ms
유클리드 1.0km clear 1425    586   0.906          0    1.44
유클리드 1.0km  rain 1284    703   0.950          0    1.22
유클리드 1.0km clear 1409    585   0.903          0    1.49
유클리드 1.0km   fog 1327    594   0.811          0    1.35
마할라노비스 3.0 clear 1425    568   0.953          0    2.27
마할라노비스 3.0  rain 1284    693   0.978          0    2.36
마할라노비스 3.0 clear 1409    570   0.947          0    2.72
마할라노비스 3.0   fog 1327    577   0.886          0    2.82

            통합 정확도  스캔당 ms
방식                        
마할라노비스 3.0   0.941   2.542
유클리드 1.0km   0.893   1.375

목표(통합 정확도 75% 이상) 달성 여부: 달성 (최고 94.1%)

[게이트 민감도] 작으면 같은 비행체가 쪼개지고, 크면 다른 비행체·오경보가 섞인다
euclid 0.3:0.300 0.6:0.642 1.0:0.906 1.5:0.972 2.5:0.964
mahal 1.5:0.411 2.0:0.675 3.0:0.953 4.0:0.989 6.0:0.983

시각 동기화 생략 시(기동 시나리오) 통합 정확도 0.881 -> 동기화 적용 0.947

[스캔 25초 융합 결과 예시: 묶음 13개]
위치( 14.85,  2.39) 센서 ['ACU-D', 'EOIR-C', 'RDR-A', 'RDR-B'] 결합 거리 [0.63, 2.08, 0.35] 정답 ['TGT05']
위치(  7.30,  9.20) 센서 ['EOIR-C', 'RDR-A', 'RDR-B'] 결합 거리 [1.19, 1.7] 정답 ['TGT04']
위치( 10.20, -2.85) 센서 ['EOIR-C', 'RDR-A', 'RDR-B'] 결합 거리 [1.55, 1.18] 정답 ['TGT00']
위치( 12.85, -0.54) 센서 ['EOIR-C', 'RDR-A', 'RDR-B'] 결합 거리 [1.58, 0.28] 정답 ['TGT01']
위치(  0.08,  1.81) 센서 ['EOIR-C', 'RDR-A', 'RDR-B'] 결합 거리 [0.8, 1.71] 정답 ['TGT03']
한계: 스캔마다 독립적으로 묶어 속도 정보를 쓰지 않으므로, 가까운 편대·교차 표적과 단일 센서 오경보를 구분하기 어렵다(미션 4에서 칼만 필터로 보완).
```
</details>
